# Лабораторная работа №3

## Трекинг объектов

### Цель

Построить трекер объекта, объединяющий детекцию и сопровождение, и сравнить не менее двух стратегий трекинга на одном видео с эталонной разметкой по количественным метрикам.

Работа итоговая для видеочасти блока: проверяется не отдельный удачный проход по ролику, а поведение трекера в сценах отказа — перекрытие, смена освещения, быстрое движение — и корректность обработки потери объекта с повторным захватом.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Библиотеки: `opencv-python`, `numpy`, `scikit-image`, `matplotlib`, `pandas`. Корреляционные трекеры CSRT и KCF доступны только в сборке `opencv-contrib-python`; в заготовке их создание обёрнуто в проверку доступности, и ноутбук исполняется целиком даже при их отсутствии.

Данные. Ноутбук работает без интернета: видео с эталонными боксами генерируется синтетически. В последовательности заложены сцены отказа:

- статичный перекрывающий объект (объект уходит за него и появляется снова) — проверка потери и повторного захвата;
- скачок освещения — проверка устойчивости шаблонных и корреляционных схем;
- мелкая колеблющаяся помеха — источник ложного захвата.

Функция чтения реального видеофайла также готова: если вы подложите собственную съёмку или последовательность MOT17 с разметкой (карточки в [resources/datasets](../../resources/datasets/README.md)), остальной код не меняется. Формат эталона — таблица `frame, x, y, w, h, visible`.

Готовыми даны: генератор видео, чтение и запись видео, метрики трекинга, визуализация траектории и IoU по кадрам, опорная стратегия сопровождения, утилиты для оптического потока и создание корреляционных трекеров. Стратегии трекинга, критерий потери, повторный захват и планирование сравнения — ваша часть.

In [ ]:
# Зависимости (при необходимости раскомментируйте):
# %pip install opencv-python numpy scikit-image matplotlib pandas
# Корреляционные трекеры CSRT/KCF требуют сборки с contrib-модулями:
# %pip install opencv-contrib-python

import platform
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage

SEED = 42
rng = np.random.default_rng(SEED)
cv2.setRNGSeed(SEED)

OUTPUT_DIR = Path("outputs_lab3")
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("python      :", platform.python_version())
print("opencv      :", cv2.__version__)
print("numpy       :", np.__version__)
print("scikit-image:", skimage.__version__)
print("pandas      :", pd.__version__)
print("SEED        :", SEED)

## 2. Краткая теоретическая справка

### 2.1. Оптический поток Лукаса—Канаде

Предполагается постоянство яркости точки при смещении:

$$I(x, y, t) = I(x + u,\ y + v,\ t + \Delta t).$$

Линеаризация даёт уравнение потока $I_x u + I_y v + I_t = 0$ — одно уравнение на две неизвестные (проблема апертуры). Лукас и Канаде добавляют допущение о постоянстве потока в окне $\Omega$ и решают переопределённую систему методом наименьших квадратов:

$$\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}
\begin{bmatrix} u \\ v \end{bmatrix} =
-\begin{bmatrix} \sum I_x I_t \\ \sum I_y I_t \end{bmatrix}.$$

Матрица слева — та же матрица вторых моментов, что и в детекторе Харриса: поток надёжно оценивается только в точках, где она хорошо обусловлена. Отсюда допущения метода: малые смещения (снимаются пирамидальной реализацией), постоянная яркость (нарушается при смене освещения), локальная согласованность движения (нарушается на границах объекта и при перекрытии).

### 2.2. Корреляционные трекеры

KCF обучает регрессор отклика по циклическим сдвигам окна вокруг объекта; циклическая структура позволяет вычислять решение через быстрое преобразование Фурье. CSRT добавляет пространственную маску надёжности, что улучшает работу с невыпуклыми объектами и частичными перекрытиями ценой скорости. Оба трекера строят модель объекта по кадру инициализации и обновляют её, поэтому при длительном перекрытии модель загрязняется фоном — это основная причина «залипания» трекера на месте перекрытия.

### 2.3. Детекция и сопровождение

Сопровождение (tracking) использует предыдущее положение и потому дёшево, но накапливает ошибку и не восстанавливается само. Детекция независима от истории, но дороже и даёт ложные срабатывания. Практическая схема: сопровождение на каждом кадре, детекция — при инициализации и при повторном захвате после потери.

Критерий потери должен быть явным: падение отклика трекера ниже порога, расхождение направления смещения точек, исчезновение переднего плана в окрестности, выход прямоугольника за кадр.

### 2.4. Метрики

Покадровый IoU прямоугольников:

$$\mathrm{IoU}_t = \frac{|B_t^{pred} \cap B_t^{gt}|}{|B_t^{pred} \cup B_t^{gt}|}, \qquad
\mathrm{Success} = \frac{1}{N}\sum_t \bigl[\mathrm{IoU}_t \ge \tau \bigr].$$

Доля потерь — доля кадров, на которых объект виден в эталоне, но трекер не выдал прямоугольника либо выдал с $\mathrm{IoU}_t < \tau$. Отдельно фиксируется число повторных захватов и задержка повторного захвата в кадрах.

## 3. Задачи

Формулировка из [методических указаний блока](README.md#лр3-трекинг-объектов):

Итоговая работа по видеопотоку: постройте трекер объекта, комбинируя детекцию (характеристические точки или вычитание фона) и сопровождение (оптический поток Лукаса–Канаде и/или корреляционный трекер CSRT/KCF). Обязательные элементы: инициализация, обработка потери объекта, повторный захват. Сравните не менее двух стратегий трекинга на одном видео с ground truth (например, из MOT17 или собственной разметки).

**Результат:** метрики трекинга (IoU по кадрам, доля потерь), сравнение стратегий, анализ сцен отказа (перекрытия, смена освещения).

Проверяемые элементы ([рубрика](../teachers-assessment/README.md)): есть инициализация, обработка потери и повторный захват; сравнение не менее двух стратегий на одном видео с ground truth; IoU по кадрам и доля потерь посчитаны.

## 4. Данные: видео со сценами отказа

Генератор тот же, что в [ДЗ6](README.md#дз6-детекция-объекта-внимания-в-видеопотоке), но с включённым перекрывающим объектом. Кадры, где объект скрыт, помечены `visible = 0`: на них корректный трекер должен сообщать о потере, а не выдавать прямоугольник.

Вторая последовательность (`sequence_hard`) отличается более быстрым движением и более ранним скачком освещения. Она нужна для проверки переносимости настроек: подбирать параметры на ней нельзя.

In [ ]:
def make_background(h, w, seed=SEED):
    """Статичный текстурированный фон. Выход: (h, w, 3) uint8 BGR."""
    generator = np.random.default_rng(seed)
    coarse = generator.normal(0.0, 1.0, (max(h // 8, 2), max(w // 8, 2), 3))
    field = cv2.resize(coarse, (w, h), interpolation=cv2.INTER_CUBIC)
    field = cv2.GaussianBlur(field, (0, 0), 3)
    field = 40.0 + 60.0 * (field - field.min()) / (float(np.ptp(field)) + 1e-9)
    gradient = np.linspace(0.0, 50.0, w, dtype=np.float32)[None, :, None]
    return np.clip(field + gradient, 0, 255).astype(np.uint8)


def make_synthetic_video(n_frames=180, size=(240, 320), *, noise_sigma=6.0,
                         object_size=(46, 34), speed=1.0, distractor=True,
                         illumination=None, occluder=(150, 60, 40, 150), seed=SEED):
    """Синтетическое видео со статичной камеры с эталонной разметкой.

    Вход:
        speed        — множитель скорости движения объекта;
        illumination — None или (frame_index, gain);
        occluder     — None или (x, y, w, h) статичной перекрывающей области.
    Выход:
        frames — список (h, w, 3) uint8 BGR;
        masks  — список (h, w) uint8 {0, 1} — видимая часть объекта;
        gt     — DataFrame: frame, x, y, w, h, visible.
    """
    h, w = size
    generator = np.random.default_rng(seed)
    background = make_background(h, w, seed)
    ow, oh = object_size

    frames, masks, rows = [], [], []
    for index in range(n_frames):
        t = (index / max(n_frames - 1, 1)) * speed
        cx = 30.0 + (w - 60.0) * (t % 1.0)
        cy = h * 0.5 + 0.20 * h * np.sin(2.0 * np.pi * 1.5 * t)
        x = int(np.clip(cx - ow / 2.0, 0, w - ow))
        y = int(np.clip(cy - oh / 2.0, 0, h - oh))

        frame = background.copy()
        if distractor:
            shift = int(round(6.0 * np.sin(2.0 * np.pi * 6.0 * t)))
            cv2.circle(frame, (int(0.12 * w) + shift, int(0.85 * h)), 7, (160, 160, 160), -1)

        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.rectangle(mask, (x, y), (x + ow, y + oh), 1, -1)
        frame[mask == 1] = (225, 225, 235)
        # Текстура на объекте: без неё оптический поток не имеет надёжных точек.
        cv2.line(frame, (x + 6, y + 6), (x + ow - 6, y + oh - 6), (90, 90, 110), 2)
        cv2.line(frame, (x + 6, y + oh - 6), (x + ow - 6, y + 6), (90, 90, 110), 2)

        visible = 1
        if occluder is not None:
            ox, oy, oww, ohh = occluder
            cv2.rectangle(frame, (ox, oy), (ox + oww, oy + ohh), (35, 35, 45), -1)
            inter_w = max(0, min(x + ow, ox + oww) - max(x, ox))
            inter_h = max(0, min(y + oh, oy + ohh) - max(y, oy))
            mask[oy:oy + ohh, ox:ox + oww] = 0
            visible = int((inter_w * inter_h) / float(ow * oh) < 0.6)

        frame = frame.astype(np.float32)
        if illumination is not None and index >= illumination[0]:
            frame = frame * float(illumination[1])
        frame = frame + generator.normal(0.0, noise_sigma, frame.shape)

        frames.append(np.clip(frame, 0, 255).astype(np.uint8))
        masks.append(mask)
        rows.append({"frame": index, "x": x, "y": y, "w": ow, "h": oh, "visible": visible})

    return frames, masks, pd.DataFrame(rows)


sequence = {}
sequence["main"] = make_synthetic_video(illumination=(120, 1.35))
sequence["hard"] = make_synthetic_video(n_frames=180, speed=1.8, noise_sigma=9.0,
                                        illumination=(60, 0.7), seed=SEED + 1)

frames, masks_gt, gt = sequence["main"]
print("main: кадров", len(frames), "| скрыт в", int((gt["visible"] == 0).sum()), "кадрах")
print("hard: кадров", len(sequence["hard"][0]),
      "| скрыт в", int((sequence["hard"][2]["visible"] == 0).sum()), "кадрах")

In [ ]:
def draw_box(image, box, color=(0, 255, 0), label=None):
    """Нарисовать прямоугольник (x, y, w, h) на копии кадра."""
    canvas = image.copy()
    if box is not None:
        x, y, w, h = [int(round(v)) for v in box]
        cv2.rectangle(canvas, (x, y), (x + w, y + h), color, 2)
        if label:
            cv2.putText(canvas, label, (x, max(12, y - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return canvas


def show_frames(frames, indices, boxes=None, title=""):
    """Показать кадры с прямоугольниками. boxes — dict {frame_index: box|None}."""
    fig, axes = plt.subplots(1, len(indices), figsize=(3.2 * len(indices), 3.4))
    for ax, index in zip(np.atleast_1d(axes), indices):
        box = boxes.get(index) if boxes else None
        ax.imshow(cv2.cvtColor(draw_box(frames[index], box), cv2.COLOR_BGR2RGB))
        ax.set_title(f"кадр {index}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def read_video_frames(path, max_frames=None, to_size=None):
    """Покадровое чтение видеофайла. Выход: список кадров BGR uint8."""
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Не удалось открыть видео: {path}")
    buffer = []
    try:
        while max_frames is None or len(buffer) < max_frames:
            ok, frame = capture.read()
            if not ok:
                break
            if to_size is not None:
                frame = cv2.resize(frame, to_size, interpolation=cv2.INTER_AREA)
            buffer.append(frame)
    finally:
        capture.release()
    return buffer


def save_video(frames, path, fps=25):
    """Записать кадры в файл (артефакт отчёта)."""
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()
    return path


gt_boxes = {int(r.frame): (r.x, r.y, r.w, r.h) for r in gt.itertuples() if r.visible}
show_frames(frames, [0, 60, 95, 125, 175], gt_boxes, "Последовательность main и эталон")

## 5. Метрики трекинга и журнал экспериментов

Считаются: средний IoU по видимым кадрам, доля успешных кадров при пороге 0.5, доля потерь, число ложных прямоугольников на кадрах перекрытия, число повторных захватов и средняя задержка повторного захвата.

Обратите внимание на трактовку кадров с `visible = 0`: там правильный ответ — отсутствие прямоугольника. Трекер, который «уверенно» продолжает вести объект за перекрытием, получает штраф.

In [ ]:
def iou_box(box_a, box_b):
    """IoU двух прямоугольников (x, y, w, h)."""
    if box_a is None or box_b is None:
        return 0.0
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    inter_w = max(0.0, min(ax + aw, bx + bw) - max(ax, bx))
    inter_h = max(0.0, min(ay + ah, by + bh) - max(ay, by))
    inter = inter_w * inter_h
    union = aw * ah + bw * bh - inter
    return 0.0 if union <= 0 else float(inter / union)


def evaluate_tracking(gt_frame, boxes, iou_threshold=0.5):
    """Метрики трекинга по эталонной разметке.

    Вход:
        gt_frame — DataFrame: frame, x, y, w, h, visible;
        boxes    — dict {frame_index: (x, y, w, h) или None};
        iou_threshold — порог успешного кадра.
    Выход:
        (metrics dict, per_frame DataFrame с колонками frame, iou, visible, tracked).

    Определения:
        mean_iou       — средний IoU по кадрам с visible = 1;
        success_rate   — доля кадров с visible = 1 и IoU >= порога;
        lost_rate      — 1 - success_rate (пропуски и грубые ошибки положения);
        ghost_rate     — доля кадров с visible = 0, на которых выдан прямоугольник;
        n_reacquire    — число переходов «нет прямоугольника -> успешный кадр»;
        mean_reacquire_delay — средняя длина серии потерь перед таким переходом.
    """
    rows = []
    for row in gt_frame.itertuples():
        predicted = boxes.get(int(row.frame))
        truth = (row.x, row.y, row.w, row.h) if row.visible else None
        value = iou_box(truth, predicted)
        rows.append({"frame": int(row.frame), "visible": int(row.visible),
                     "tracked": int(predicted is not None), "iou": value})
    per_frame = pd.DataFrame(rows)

    visible = per_frame[per_frame["visible"] == 1]
    hidden = per_frame[per_frame["visible"] == 0]
    success = (visible["iou"] >= iou_threshold)

    n_reacquire, delays, gap = 0, [], 0
    for row in per_frame.itertuples():
        ok = row.visible == 1 and row.iou >= iou_threshold
        if ok:
            if gap > 0:
                n_reacquire += 1
                delays.append(gap)
            gap = 0
        else:
            gap += 1

    metrics = {
        "mean_iou": round(float(visible["iou"].mean()) if len(visible) else 0.0, 4),
        "success_rate": round(float(success.mean()) if len(visible) else 0.0, 4),
        "lost_rate": round(1.0 - (float(success.mean()) if len(visible) else 0.0), 4),
        "ghost_rate": round(float(hidden["tracked"].mean()) if len(hidden) else 0.0, 4),
        "n_reacquire": int(n_reacquire),
        "mean_reacquire_delay": round(float(np.mean(delays)) if delays else 0.0, 2),
    }
    return metrics, per_frame


def plot_iou_curve(curves, title="IoU по кадрам"):
    """Наложить кривые IoU нескольких стратегий. curves: {name: per_frame DataFrame}."""
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for name, per_frame in curves.items():
        ax.plot(per_frame["frame"], per_frame["iou"], label=name, linewidth=1.2)
    reference = next(iter(curves.values()))
    hidden = reference[reference["visible"] == 0]["frame"].to_numpy()
    if len(hidden):
        ax.fill_between(reference["frame"], 0, 1,
                        where=reference["visible"].to_numpy() == 0,
                        color="0.85", label="объект скрыт")
    ax.set_xlabel("кадр")
    ax.set_ylabel("IoU")
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()


RUNS = []


def log_run(**fields):
    """Добавить запись в журнал экспериментов."""
    record = {"run_id": len(RUNS), **fields}
    RUNS.append(record)
    return record


def runs_table(columns=None, sort_by=None):
    if not RUNS:
        return pd.DataFrame()
    frame = pd.DataFrame(RUNS)
    if columns:
        frame = frame[[c for c in columns if c in frame.columns]]
    if sort_by:
        frame = frame.sort_values(sort_by)
    return frame.reset_index(drop=True)


def save_runs(path=OUTPUT_DIR / "runs.csv"):
    runs_table().to_csv(path, index=False)
    return path

## 6. Опорная стратегия сопровождения

Baseline — поиск неизменяемого шаблона в окне вокруг предыдущего положения. Стратегия намеренно неполная: у неё нет ни адаптации модели, ни повторного захвата. Она задаёт нижнюю границу качества и показывает, что именно должны добавить ваши стратегии.

Инициализация во всех стратегиях выполняется одинаково — по эталонному прямоугольнику первого кадра. Если вы инициализируете трекер детектором (вычитание фона или характеристические точки), зафиксируйте это как отдельную конфигурацию: инициализация влияет на метрики не меньше, чем сам трекер.

In [ ]:
def track_template_baseline(frames, init_box, *, search_margin=25, score_threshold=0.55):
    """Опорная стратегия: сопоставление фиксированного шаблона в окне поиска.

    Вход:
        frames        — список кадров BGR uint8;
        init_box      — (x, y, w, h) на нулевом кадре;
        search_margin — расширение окна поиска вокруг предыдущего положения;
        score_threshold — порог отклика, ниже которого кадр считается потерянным.
    Выход:
        (boxes dict {frame: box|None}, scores dict {frame: float}).

    Ограничения (осознанные): шаблон не обновляется, потеря не обрабатывается,
    повторного захвата нет.
    """
    gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
    x, y, w, h = [int(v) for v in init_box]
    template = gray[0][y:y + h, x:x + w]
    boxes = {0: (x, y, w, h)}
    scores = {0: 1.0}
    last = (x, y, w, h)

    for index in range(1, len(gray)):
        height, width = gray[index].shape
        sx = max(0, last[0] - search_margin)
        sy = max(0, last[1] - search_margin)
        ex = min(width, last[0] + w + search_margin)
        ey = min(height, last[1] + h + search_margin)
        window = gray[index][sy:ey, sx:ex]
        if window.shape[0] < h or window.shape[1] < w:
            boxes[index], scores[index] = None, 0.0
            continue
        response = cv2.matchTemplate(window, template, cv2.TM_CCOEFF_NORMED)
        _, score, _, location = cv2.minMaxLoc(response)
        scores[index] = float(score)
        if score < score_threshold:
            boxes[index] = None
            continue
        last = (sx + location[0], sy + location[1], w, h)
        boxes[index] = last
    return boxes, scores


init_box = (int(gt.iloc[0]["x"]), int(gt.iloc[0]["y"]),
            int(gt.iloc[0]["w"]), int(gt.iloc[0]["h"]))

start = time.perf_counter()
baseline_boxes, baseline_scores = track_template_baseline(frames, init_box)
elapsed = time.perf_counter() - start

baseline_metrics, baseline_curve = evaluate_tracking(gt, baseline_boxes)
log_run(sequence="main", strategy="template_baseline",
        params="search_margin=25, score_threshold=0.55, update=none, reacquire=none",
        fps=round(len(frames) / elapsed, 1), seconds=round(elapsed, 3),
        seed=SEED, **baseline_metrics)

print("Baseline:", baseline_metrics)
plot_iou_curve({"template_baseline": baseline_curve}, "Baseline: IoU по кадрам")
show_frames(frames, [0, 60, 95, 125, 175], baseline_boxes, "Baseline: результат сопровождения")

## 7. Инструменты для ваших стратегий

Ниже готовы утилиты, на которых собираются обе обязательные стратегии: отбор точек внутри прямоугольника, пирамидальный Лукас—Канаде, оценка смещения по устойчивым точкам и создание корреляционного трекера с проверкой доступности `opencv-contrib-python`.

Утилиты не являются трекером: в них нет ни критерия потери, ни повторного захвата, ни обновления модели.

In [ ]:
def points_in_box(gray, box, max_points=80, quality=0.01, min_distance=3):
    """Характеристические точки внутри прямоугольника.

    Вход:  gray (H, W) uint8; box (x, y, w, h).
    Выход: (N, 1, 2) float32 в координатах всего кадра или None.
    """
    x, y, w, h = [int(v) for v in box]
    mask = np.zeros(gray.shape, dtype=np.uint8)
    mask[max(y, 0):y + h, max(x, 0):x + w] = 255
    return cv2.goodFeaturesToTrack(gray, maxCorners=max_points, qualityLevel=quality,
                                   minDistance=min_distance, mask=mask)


LK_PARAMS = dict(winSize=(21, 21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))


def lucas_kanade(prev_gray, next_gray, points, back_check_px=1.0):
    """Пирамидальный Лукас—Канаде с прямой-обратной проверкой.

    Вход:  prev_gray, next_gray — (H, W) uint8; points — (N, 1, 2) float32.
    Выход: (good_prev (M, 2), good_next (M, 2)) — точки, прошедшие проверку.
    """
    if points is None or len(points) == 0:
        return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)
    forward, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, points, None, **LK_PARAMS)
    backward, _, _ = cv2.calcOpticalFlowPyrLK(next_gray, prev_gray, forward, None, **LK_PARAMS)
    error = np.linalg.norm(points.reshape(-1, 2) - backward.reshape(-1, 2), axis=1)
    keep = (status.reshape(-1) == 1) & (error < back_check_px)
    return points.reshape(-1, 2)[keep], forward.reshape(-1, 2)[keep]


def median_shift(prev_points, next_points):
    """Медианное смещение набора точек. Выход: (dx, dy) или None."""
    if len(prev_points) == 0:
        return None
    delta = next_points - prev_points
    return float(np.median(delta[:, 0])), float(np.median(delta[:, 1]))


def shift_box(box, delta, shape):
    """Сдвинуть прямоугольник и обрезать по границам кадра."""
    if delta is None:
        return None
    x, y, w, h = box
    height, width = shape[:2]
    nx = float(np.clip(x + delta[0], 0, width - w))
    ny = float(np.clip(y + delta[1], 0, height - h))
    return (nx, ny, w, h)


# Демонстрация на двух соседних кадрах (рельсы).
prev_gray = cv2.cvtColor(frames[10], cv2.COLOR_BGR2GRAY)
next_gray = cv2.cvtColor(frames[11], cv2.COLOR_BGR2GRAY)
p_prev, p_next = lucas_kanade(prev_gray, next_gray, points_in_box(prev_gray, gt_boxes[10]))
print("Точек прошло прямую-обратную проверку:", len(p_prev),
      "| медианное смещение:", median_shift(p_prev, p_next))

In [ ]:
def create_tracker(name):
    """Создать корреляционный трекер OpenCV.

    Вход:  name — 'csrt' | 'kcf' | 'mil'.
    Выход: объект трекера с методами init(frame, box) и update(frame).

    CSRT и KCF входят в contrib-модули: при сборке без них поднимается
    RuntimeError с указанием, что установить. MIL доступен в базовой сборке
    и может служить запасным вариантом (его результаты не эквивалентны).
    """
    key = name.lower()
    legacy = getattr(cv2, "legacy", None)
    names = {"csrt": "TrackerCSRT_create", "kcf": "TrackerKCF_create",
             "mil": "TrackerMIL_create"}
    if key not in names:
        raise ValueError(f"Неизвестный трекер: {name}")
    factories = [getattr(cv2, names[key], None), getattr(legacy, names[key], None)]
    errors = []
    for factory in factories:
        if factory is None:
            continue
        try:
            return factory()
        except cv2.error as exc:
            errors.append(str(exc))
    raise RuntimeError(
        f"Трекер {name.upper()} недоступен в текущей сборке OpenCV {cv2.__version__}. "
        "Установите сборку с contrib-модулями: pip install opencv-contrib-python "
        "(и удалите opencv-python, чтобы пакеты не конфликтовали). "
        + (" Детали: " + errors[0] if errors else "")
    )


# Проверка доступности трекеров в текущем окружении.
AVAILABLE_TRACKERS = []
for name in ["csrt", "kcf", "mil"]:
    try:
        create_tracker(name)
    except (RuntimeError, ValueError) as exc:
        print(f"{name.upper()}: недоступен. {exc}")
    else:
        AVAILABLE_TRACKERS.append(name)
        print(f"{name.upper()}: доступен")

print("Доступные корреляционные трекеры:", AVAILABLE_TRACKERS)

# Рельсы: короткий прогон доступного трекера на первых кадрах.
if AVAILABLE_TRACKERS:
    demo_name = AVAILABLE_TRACKERS[0]
    tracker = create_tracker(demo_name)
    tracker.init(frames[0], tuple(int(v) for v in init_box))
    demo_boxes = {0: init_box}
    for index in range(1, 30):
        ok, box = tracker.update(frames[index])
        demo_boxes[index] = tuple(box) if ok else None
    print(f"Демонстрация {demo_name.upper()}: кадр 29 ->", demo_boxes[29])
else:
    print("Корреляционные трекеры недоступны: стратегию на их основе замените "
          "второй схемой на оптическом потоке и укажите это в отчёте.")

## 8. Стратегии трекинга

Стратегия — это полный цикл: инициализация, сопровождение, обнаружение потери, повторный захват. Приведите обе стратегии к одному интерфейсу, иначе сравнение придётся делать вручную и оно будет несопоставимым.

Требования к повторному захвату: он должен быть независим от последнего положения трекера (иначе это не захват, а продолжение дрейфа). Разумные источники кандидатов — вычитание фона из [ДЗ6](README.md#дз6-детекция-объекта-внимания-в-видеопотоке) или сопоставление характеристических точек из [ДЗ4](README.md#дз4-методы-детекции-характеристических-точек).

Ловушка: повторный захват по эталонной разметке (`gt`) недопустим — это утечка эталона в алгоритм. Эталон используется только для оценки.

In [ ]:
def track(frames, init_box, strategy, **params):
    """Единый интерфейс стратегии трекинга.

    Контракт.
    Вход:
        frames   — список кадров BGR uint8 в исходном порядке;
        init_box — (x, y, w, h) на нулевом кадре;
        strategy — 'lk' | 'csrt' | 'kcf' | 'lk+detect' | ... (ваши имена);
        params   — параметры стратегии (порог потери, параметры повторного
                   захвата, частота обновления модели и т. п.).
    Выход:
        (boxes, info): boxes — dict {frame_index: (x, y, w, h) или None};
                       info  — dict с полями 'state' (список состояний по кадрам:
                       'tracking' | 'lost' | 'reacquired'), 'score' (отклик или
                       иной показатель уверенности по кадрам).

    Требования:
        1) кадры обрабатываются последовательно; эталон gt внутрь не передаётся;
        2) критерий потери задан явно и записан в docstring вашей реализации;
        3) после потери прямоугольник не выдаётся (boxes[i] = None) до успешного
           повторного захвата;
        4) повторный захват выполняется независимым детектором;
        5) при недоступности CSRT/KCF стратегия должна корректно сообщать об этом
           через create_tracker, а не падать с невнятной ошибкой.
    """
    raise NotImplementedError("TODO (задание 3): реализуйте стратегии трекинга")

In [ ]:
# TODO (задание 3): реализуйте и прогоните не менее двух стратегий на
# последовательности 'main'.
#
# Обязательный минимум:
#   1) стратегия на оптическом потоке Лукаса—Канаде (points_in_box + lucas_kanade
#      + median_shift + shift_box), с переинициализацией набора точек;
#   2) стратегия на корреляционном трекере (CSRT или KCF), а при их отсутствии —
#      вторая схема с иным критерием потери и иным механизмом повторного захвата;
#   3) в обеих: явный критерий потери и повторный захват независимым детектором.
#
# Для каждой стратегии:
#   boxes, info = track(frames, init_box, "lk", ...)
#   metrics, curve = evaluate_tracking(gt, boxes)
#   log_run(sequence="main", strategy="lk", params="...", fps=..., **metrics)
#
# Сохраните кривые IoU в словарь curves для общего графика.

curves = {"template_baseline": baseline_curve}
runs_table()

In [ ]:
# TODO (задание 3): проверка переносимости и сцены отказа.
#
#   1) прогоните ТЕ ЖЕ конфигурации (без подстройки) на последовательности 'hard'
#      и сравните падение метрик со сдвигом относительно 'main';
#   2) отдельно посчитайте метрики на трёх участках: до перекрытия, во время
#      перекрытия и после него; и отдельно — до и после скачка освещения;
#   3) найдите кадр, на котором каждая стратегия теряет объект, и сохраните
#      иллюстрацию: кадр потери, кадр во время потери, кадр повторного захвата.
#
# frames_hard, masks_hard, gt_hard = sequence["hard"]

pass

## Отчёт

### Сводные таблицы

Обязательны:

1. «стратегия × последовательность → mean IoU, success rate, доля потерь, ghost rate, число повторных захватов, задержка захвата, fps»;
2. таблица по участкам сцены (до/во время/после перекрытия; до/после смены освещения);
3. общий график IoU по кадрам для всех стратегий с отмеченными интервалами перекрытия.

In [ ]:
frame = runs_table()

if frame.empty:
    print("Журнал пуст: выполните разделы 6 и 8.")
else:
    columns = [c for c in ["sequence", "strategy", "params", "mean_iou", "success_rate",
                           "lost_rate", "ghost_rate", "n_reacquire",
                           "mean_reacquire_delay", "fps"] if c in frame.columns]
    display(frame[columns].round(3))

plot_iou_curve(curves, "Сравнение стратегий: IoU по кадрам")
save_runs()

### Выводы

**Наблюдения** (измеренные факты со ссылкой на строки журнала):

-

**Интерпретация** (почему конкретная стратегия теряет объект именно в этих условиях; какое допущение метода нарушается — постоянство яркости, малость смещения, целостность модели объекта):

-

**Выводы и их границы** (какие последовательности, какое разрешение, какие диапазоны параметров; что не проверялось: несколько объектов, движущаяся камера, длительные перекрытия):

-

**Анализ сцен отказа**: обязательно разберите перекрытие и смену освещения; для каждой стратегии покажите кадр потери, поведение во время потери и кадр повторного захвата.

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока):

6. Как модель фона MOG2 адаптируется к изменению освещения и чем это опасно?
7. Какие допущения делает оптический поток Лукаса–Канаде?

Дополнительно:

- Покажите кадр, где ваш трекер теряет объект. Почему это происходит и как реализован повторный захват (вопрос к защите)?
- Чем отличается дрейф трекера от потери и как отличить одно от другого по метрикам?
- Почему прямая-обратная проверка точек снижает число ложных смещений?

## Чек-лист перед сдачей

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`, в том числе при отсутствии `opencv-contrib-python`.
- [ ] Указаны ФИО, группа, номер работы, источники данных.
- [ ] Seed и версии библиотек зафиксированы и выведены.
- [ ] Реализованы инициализация, обнаружение потери и повторный захват; критерий потери описан явно.
- [ ] Повторный захват не использует эталонную разметку.
- [ ] Сравнены не менее двух стратегий на одном видео с ground truth.
- [ ] Посчитаны IoU по кадрам и доля потерь; есть график IoU по кадрам.
- [ ] Проверена переносимость конфигураций на вторую последовательность.
- [ ] Разобраны сцены отказа: перекрытие и смена освещения, с иллюстрациями.
- [ ] Есть сводные таблицы; журнал сохранён (`outputs_lab3/runs.csv`).
- [ ] Наблюдения, интерпретация и выводы разделены; указаны ограничения.